In [ ]:
# frame2d.jl
# Chương trình phân tích hệ khung phẳng (compact 2D frame analysis)
# Phiên bản nhỏ gọn: phần tử beam-column 2-node, DOF trên mỗi nút = [u, v, θ]
using LinearAlgebra

# local stiffness (6x6) cho phần tử beam-column
function ke_local(L, E, A, I)
    EA_L = E*A / L
    EI = E*I
    k = zeros(6,6)
    k[1,1] =  EA_L
    k[1,4] = -EA_L
    k[4,1] = -EA_L
    k[4,4] =  EA_L

    k[2,2] =  12*EI / L^3
    k[2,3] =   6*EI / L^2
    k[2,5] = -12*EI / L^3
    k[2,6] =   6*EI / L^2

    k[3,2] =   6*EI / L^2
    k[3,3] =   4*EI / L
    k[3,5] =  -6*EI / L^2
    k[3,6] =   2*EI / L

    k[5,2] = -12*EI / L^3
    k[5,3] =  -6*EI / L^2
    k[5,5] =  12*EI / L^3
    k[5,6] =  -6*EI / L^2

    k[6,2] =   6*EI / L^2
    k[6,3] =   2*EI / L
    k[6,5] =  -6*EI / L^2
    k[6,6] =   4*EI / L

    return Symmetric(k) # đảm bảo tính đối xứng
end

# chuyển từ global -> local: ma trận 6x6
function T_matrix(dx, dy)
    L = hypot(dx, dy)
    c = dx / L
    s = dy / L
    R = [c s 0; -s c 0; 0 0 1]
    T = zeros(6,6)
    T[1:3,1:3] = R
    T[4:6,4:6] = R
    return T
end

# Hàm phân tích hệ khung
# nodes: Nx2 array of coordinates
# elems: Mx2 array of node indices (1-based)
# E, A, I: scalar hoặc vectors kích thước M
# loads: Nx3 array (Fx, Fy, M) tổng lực ngoại tại tại nút (positive y lên, M positive ccw)
# bcs: Nx3 Bool (true = cố định DOF)
function analyze(nodes::Array{<:Real,2}, elems::Array{Int,2},
                 E, A, I, loads=zeros(size(nodes,1),3), bcs=zeros(Bool,size(nodes,1),3))

    N = size(nodes,1)
    M = size(elems,1)
    ndof = 3*N
    K = zeros(ndof, ndof)
    F = vec(permutedims(loads)) # order nodes: [u1,v1,θ1,u2,v2,θ2,...]

    # ensure vector or broadcast to elements
    Elist = isa(E, Number) ? fill(E, M) : E
    Alist = isa(A, Number) ? fill(A, M) : A
    Ilist = isa(I, Number) ? fill(I, M) : I

    for (idx, e) in enumerate(eachrow(elems))
        n1, n2 = e
        x1, y1 = nodes[n1, :]
        x2, y2 = nodes[n2, :]
        dx, dy = x2-x1, y2-y1
        L = hypot(dx, dy)
        ke = ke_local(L, Elist[idx], Alist[idx], Ilist[idx])
        T = T_matrix(dx, dy)
        kg = T' * ke * T
        dof_map = vcat( (3*(n1-1)+1):(3*(n1-1)+3), (3*(n2-1)+1):(3*(n2-1)+3) )
        for i in 1:6, j in 1:6
            K[dof_map[i], dof_map[j]] += kg[i,j]
        end
    end

    # Apply boundary conditions: fixed DOFs removed
    fixed = vec(permutedims(bcs))
    free_dofs = findall(!, fixed)
    Kff = K[free_dofs, free_dofs]
    Ff = F[free_dofs]

    u = zeros(ndof)
    u[free_dofs] = Kff \ Ff

    # reactions
    reactions = K * u - F

    # element internal forces (local)
    elem_forces = Vector{Vector{Float64}}(undef, M)
    for (idx, e) in enumerate(eachrow(elems))
        n1, n2 = e
        dof_map = vcat( (3*(n1-1)+1):(3*(n1-1)+3), (3*(n2-1)+1):(3*(n2-1)+3) )
        ue = u[dof_map]
        x1, y1 = nodes[n1, :]
        x2, y2 = nodes[n2, :]
        dx, dy = x2-x1, y2-y1
        T = T_matrix(dx, dy)
        ul = T * ue            # displacements in local coords
        L = hypot(dx, dy)
        ke = ke_local(L, Elist[idx], Alist[idx], Ilist[idx])
        fl = ke * ul           # internal nodal forces in local coords
        elem_forces[idx] = fl
    end

    return (u = u, reactions = reactions, elem_forces = elem_forces, K = K)
end

# Ví dụ nhanh: dầm hình L = 1m, 2 nút, node1 cố định, node2 chịu lực hướng xuống 1000N
if abspath(PROGRAM_FILE) == @__FILE__
    nodes = [0.0 0.0; 1.0 0.0]
    elems = [1 2]
    E = 210e9
    A = 0.01
    I = 8.333e-6
    loads = zeros(2,3)
    loads[2,2] = -1000.0  # Fy tại node 2
    bcs = falses(2,3)
    bcs[1,:] .= true      # node 1 cố định hoàn toàn
    res = analyze(nodes, elems, E, A, I, loads, bcs)
    println("Nội lực phần tử (local) tại node đầu/cuối:")
    for (i, f) in enumerate(res.elem_forces)
        println("Elem $i: ", round.(f; sigdigits=6))
    end
    println("Độ dời nút (u,v,θ):")
    for i in 1:size(nodes,1)
        d = res.u[(3*(i-1)+1):(3*(i-1)+3)]
        println("Node $i: ", round.(d; sigdigits=6))
    end
    println("Phản lực (Fx,Fy,M) tại nút:")
    for i in 1:size(nodes,1)
        r = res.reactions[(3*(i-1)+1):(3*(i-1)+3)]
        println("Node $i: ", round.(r; sigdigits=6))
    end
end